# Default of Credit Card Clients — Feature Engineering Deep Dive

In [ ]:

from pathlib import Path
import sys

def _find_repo_root() -> Path:
    starts = [Path.cwd().resolve()]
    # Jupyter may set cwd to the notebook folder; also walk from this file if present
    try:
        starts.append(Path(__file__).resolve().parent)  # type: ignore[name-defined]
    except NameError:
        pass
    for start in starts:
        for p in [start, *start.parents]:
            if (p / "general_pipeline").is_dir() and (p / "hyperack_exp").is_dir():
                return p
    raise RuntimeError(
        "Could not find repo root containing general_pipeline/. "
        "Open the notebook from the R&D repo or set the kernel cwd to the repo root."
    )

ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "hyperack_exp") not in sys.path:
    sys.path.insert(0, str(ROOT / "hyperack_exp"))
print("REPO ROOT:", ROOT)
from general_pipeline.playbook.ladder import load_raw_xy
from general_pipeline.playbook.features import build_feature_matrix
from shared.protocol import evaluate
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pandas as pd

KEY = 'credit_default'
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMClassifier(random_state=42, verbosity=-1)),
])
rows = []
for mode in ['safe', 'unsafe']:
    Xtr, ytr, Xte, yte, meta = load_raw_xy(KEY, mode)
    for stage in ['raw', 'logs', 'ratios', 'interactions', 'full_fe', 'selected']:
        A, B, fe = build_feature_matrix(Xtr, ytr, Xte, stage=stage)
        m = evaluate(model, A, ytr, B, yte)
        rows.append({'mode': mode, 'stage': stage, 'feats': A.shape[1], 'roc_auc': m['roc_auc'], 'f1': m['f1']})
pd.DataFrame(rows).sort_values(['mode', 'roc_auc'], ascending=[True, False])
